In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import numpy as np

# Permitir importar desde la carpeta 'src'
sys.path.append(os.path.abspath('../'))

In [ ]:
from src.data.load_data import parse_summary_file, load_edf_signal, generate_labels_for_file
import matplotlib.pyplot as plt

# 1. Parsear el TXT de anotaciones del Paciente 1
summary_path = "../data/raw/chb01/chb01-summary.txt"
diccionario_crisis = parse_summary_file(summary_path)

# 2. Elegir el archivo que sabemos que tiene una crisis
archivo_test = "chb01_03.edf"
edf_path = f"../data/raw/chb01/{archivo_test}"

# 3. Cargar la señal biomédica real desde el EDF usando MNE
data_cruda, fs, nombres_canales = load_edf_signal(edf_path)

# 4. Generar el vector y de etiquetas reales usando el TXT parseado automáticamente
tiempos_crisis = diccionario_crisis[archivo_test] # Extrae [(2996, 3036)]
y_muestras = generate_labels_for_file(data_cruda.shape[1], fs, tiempos_crisis)

print(f"=== REPORTE DE CARGA REAL ===")
print(f"Archivo: {archivo_test} cargado con éxito.")
print(f"Frecuencia de muestreo: {fs} Hz")
print(f"Forma de la matriz de EEG real: {data_cruda.shape} (canales, muestras totales)")
print(f"Cantidad de muestras en estado de crisis real: {sum(y_muestras)} muestras (~{sum(y_muestras)/fs} segundos)")

# 5. GRAFICAR UN CANAL REAL EN EL INSTANTE DE LA CRISIS
# Graficamos 10 segundos justo cuando arranca la crisis (segundo 2996)
canal_interes = 0 # Primer canal del montaje
inicio_grafico = int(2995 * fs)
fin_grafico = int(3005 * fs)
tiempo_eje = np.linspace(2995, 3005, fin_grafico - inicio_grafico)

plt.figure(figsize=(12, 4))
plt.plot(tiempo_eje, data_cruda[canal_interes, inicio_grafico:fin_grafico], color='black', lw=0.7)
plt.axvspan(2996, 3005, color='red', alpha=0.2, label='Crisis Epiléptica Anotada')
plt.title(f"EEG Real - Paciente chb01 - Canal: {nombres_canales[canal_interes]}")
plt.xlabel("Tiempo (segundos)")
plt.ylabel("Amplitud ($\mu$V)")
plt.legend()
plt.grid(True)
plt.show()